In [ ]:
## Magic to reload modules between cells
%load_ext autoreload
%autoreload 2

## GPU setup
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.device(device)
SEED=12345
_=torch.manual_seed(SEED)

In [ ]:
## Point to the right inputs

## Don't use the full dataset...
nevents = 50000

## Where to get the trained models from
state_file_dir = "$PSCRATCH" 
state_file_name = "state_GENIE10cNuMIME_CCCONT256_ResNet50v1baselinev2NORM0max_PROJ10twobn_EXP4096_lars0.1WGT1E-6HEAD1_512_50_1AUGv0l_2M_N2_NULARVICReg_LRSCALE0.1_STEMNORM1_FINAL512.pth"
file_path = f"{state_file_dir}/{state_file_name}"

## The dataset used for training
root_data_dir = "$PSCRATCH/NULARBOX"
data_set = "GENIE10c_NuMIME_CCCONT256"
data_dir = f"{root_data_dir}/{data_set}"

In [ ]:
## Get the dataset
from larch.analysis.dataset_utils import get_dataset
from larch.datasets.nularbox.augmentations_2d import get_transform
from larch.analysis.model_utils import get_models_from_checkpoint

encoder, heads, args = get_models_from_checkpoint(file_name)
print("Loaded:", file_name)

## The the augmentation set used in training
nom_transform = get_transform(256, "no_aug", 1.0)
aug_transform = get_transform(256, args.aug_type, 1.0)

## Load the datasets
nom_dataset, nom_loader = get_dataset(data_dir, nevents, nom_transform)
aug_dataset, aug_loader = get_dataset(data_dir, nevents, aug_transform)

In [ ]:
## Pass the images through the encoder
from larch.analysis.dataset_utils import image_loop, reorder_clusters

## Load both sets of events
nom_processed  = image_loop(encoder, heads, nom_loader, device, return_hidden=True, detailed_info=True)
aug1_processed = image_loop(encoder, heads, aug_loader, device, return_hidden=True, detailed_info=True)
aug2_processed = image_loop(encoder, heads, aug_loader, device, return_hidden=True, detailed_info=True)

In [ ]:
from larch.analysis.geometry_utils import plot_similarity_distributions

plot_similarity_distributions(nom_processed['encoder'],
                              aug1_processed['encoder'],
                              aug2_processed['encoder'],
                              centering=True, bins=100)
plot_similarity_distributions(nom_processed['encoder'],
                              aug1_processed['encoder'],
                              aug2_processed['encoder'],
                              centering=False, bins=100)

In [ ]:
from larch.analysis.geometry_utils import cosine_spectrum, plot_spectrum
nom_cosine_eigvals = cosine_spectrum(nom_processed['encoder'][:10000])
plot_spectrum(nom_cosine_eigvals)
aug_cosine_eigvals = cosine_spectrum(aug1_processed['encoder'][:10000])
plot_spectrum(aug_cosine_eigvals)
